# D-05. 두 임베딩 점수 합치기 — MLP + FCN

D-04 결과: 두 임베딩이 서로 반대쪽 재질을 잡는다 (proto AUROC)

| | 조용한 재질 (carpet·soft·wood) | 덜컹거리는 재질 (concrete·tiles) |
|---|---|---|
| MLP | 0.398 | **0.697** |
| FCN | **0.651** | 0.448 |

→ 두 점수를 합치면 양쪽 그룹을 다 잡을 수 있는가?

방법 (`ood.combine_fold`)
- 모델마다 검증 녹화를 뺀 학습 녹화로 프로토타입을 만들고, 검증 녹화 점수의 중앙값·사분위 범위(IQR)로 `(점수 − 중앙값) / IQR` 표준화 → 두 모델의 크기를 맞춤
- `mean`: 두 표준화 점수의 평균 / `max`: 둘 중 큰 값 (한쪽만 낯설다고 해도 미등록)
- τ도 같은 검증 녹화에서 알려진 재질 95% 통과 값. 테스트 구간은 기준을 정하는 데 쓰지 않는다
- 단일 모델도 같은 방식으로 계산해 함께 표시 (D-04보다 학습 녹화가 검증 녹화만큼 적어 수치가 조금 다를 수 있음)

실행: Colab, CPU로 충분. 준비는 D-04와 같다

## 0. 환경 준비

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = "main"   # PR 브랜치에서 시험할 때만 그 브랜치 이름으로

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/ML-imu").exists():
        !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
    os.chdir("/content/ML-imu")
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    !pip install -q -r requirements.txt
    !python data/download.py

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path += [str(ROOT / "src"), str(ROOT / "notebooks" / "D_ood")]
print(ROOT)

In [ ]:
# 임베딩 찾기: 내 드라이브의 폴더 → 없으면 MyDrive/ 맨 위의 zip을 Colab 안에 푼다
# 공유받은 폴더는 '공유 문서함' → 오른쪽 클릭 → '바로가기 추가' → 내 드라이브 에 넣어야 보인다
MODELS = ["mlp", "fcn", "fcn-center"]
COMBOS = [("mlp", "fcn"), ("mlp", "fcn-center")]
ZIP_PATTERNS = ["ML-imu_embeddings*.zip", "*embeddings*.zip", "*fcn*.zip"]

if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    OUT = MYDRIVE / "ML-imu_D"
    SEARCH = [MYDRIVE / "ML-imu_embeddings", MYDRIVE / "ML-imu" / "embeddings", MYDRIVE / "embeddings"]
else:
    MYDRIVE = None
    OUT = ROOT / "results"
    SEARCH = []
LOCAL = ROOT / "embeddings" / "all"
SEARCH += [LOCAL, ROOT / "embeddings" / "mlp"]
OUT.mkdir(parents=True, exist_ok=True)


def find_dir(model):
    for d in SEARCH:
        if (d / f"emb_{model}_carpet_0.npy").exists():
            return d
    return None


missing = [m for m in MODELS if find_dir(m) is None]
if missing and MYDRIVE is not None:
    zips = sorted({z for p in ZIP_PATTERNS for z in MYDRIVE.glob(p)})
    LOCAL.mkdir(parents=True, exist_ok=True)
    for zp in zips:
        with zipfile.ZipFile(zp) as zf:
            for info in zf.infolist():
                name = Path(info.filename).name
                if any(name.startswith(f"{k}_{m}_") for k in ("emb", "logits") for m in missing):
                    (LOCAL / name).write_bytes(zf.read(info))
        print("압축 해제:", zp.name)

EMB = {m: find_dir(m) for m in MODELS}
for m, d in EMB.items():
    n = len(list(d.glob(f"emb_{m}_*.npy"))) if d else 0
    print(f"{m:<11} {n:3d}개 임베딩  ←  {d}")
assert all(EMB.values()), "임베딩을 못 찾은 모델이 있어요. 위 경로를 확인하세요."

## 1. 데이터

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset, CHANNEL_NAMES
from splits import load_material_folds
from evaluate import log_result
from ood import combine_fold, DIFFICULTY

X, y, rec = load_dataset("data")
folds = load_material_folds()
vib = X[:, :, CHANNEL_NAMES.index("acc_z")].std(axis=1)

QUIET = ["carpet", "soft_pvc", "soft_tiles", "wood"]
order = QUIET + ["fine_concrete", "concrete", "hard_tiles", "tiled"]
print(X.shape, len(folds), "개 split")

## 2. 147 split 실행

In [ ]:
def load_z(model, held_out, fold):
    return np.load(EMB[model] / f"emb_{model}_{held_out}_{fold}.npy")

rows = []
for combo in COMBOS:
    for s in folds:
        zs = {m: load_z(m, s["held_out"], s["fold"]) for m in combo}
        for r in combine_fold(zs, y, rec, s, vib=vib):
            rows.append({"combo": "+".join(combo), "heldout": s["held_out"], "fold": s["fold"], **r})
df = pd.DataFrame(rows)
df["group"] = np.where(df.heldout.isin(QUIET), "quiet", "loud")
df.to_csv(OUT / "D_combine_folds.csv", index=False)
print(len(df), "행")

## 3. 결과 표

In [ ]:
for combo in df.combo.unique():
    d = df[df.combo == combo]
    methods = list(dict.fromkeys(d.method))   # 단일 모델 → mean → max 순서
    t = (d.groupby("method")
         .agg(auroc=("auroc", "mean"), fpr95=("fpr95", "mean"), rho_vib=("rho_vib", "mean"),
              known_pass=("known_pass", "mean"), unknown_reject=("unknown_reject", "mean"))
         .join(d.pivot_table(index="method", columns="group", values="auroc")[["quiet", "loud"]]
               .add_prefix("auroc_"))
         .loc[methods])
    print(f"=== {combo}")
    display(t.round(3))
    print("빠진 재질별 AUROC")
    display(d.pivot_table(index="heldout", columns="method", values="auroc").loc[order, methods].round(3))
    print("빠진 재질별 미등록 거부율 (τ)")
    display(d.pivot_table(index="heldout", columns="method", values="unknown_reject").loc[order, methods].round(3))

## 4. 그림

In [ ]:
fig, axes = plt.subplots(1, len(COMBOS), figsize=(7 * len(COMBOS), 4), squeeze=False)
for ax, combo in zip(axes[0], df.combo.unique()):
    d = df[df.combo == combo]
    methods = list(dict.fromkeys(d.method))
    (d.pivot_table(index="method", columns="group", values="auroc").loc[methods, ["quiet", "loud"]]
     .rename(columns={"quiet": "quiet (carpet, soft, wood)", "loud": "loud (concrete, tiles)"})
     .plot.bar(ax=ax, width=0.8))
    ax.axhline(0.5, color="gray", ls="--", lw=1)
    ax.set_ylim(0, 1); ax.set_ylabel("AUROC"); ax.set_xlabel(""); ax.set_title(combo)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig(OUT / "D_combine.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. results.csv 기록 (합친 점수만)

In [ ]:
comb = df[df.method.str.startswith(("mean(", "max("))]
rows = comb.groupby(["method", "heldout"])[["auroc", "fpr95"]].mean().reset_index()
for _, r in rows.iterrows():
    log_result("D", f"{r.method}+proto", {"auroc": r.auroc, "fpr95": r.fpr95},
               setting=f"heldout={r.heldout}", fold="mean",
               note=f"{DIFFICULTY.get(r.heldout, 'mid')}, val-IQR 표준화 후 합침 (D-05)")

import shutil
shutil.copy(ROOT / "results" / "results.csv", OUT / "results_with_D05.csv")
print(len(rows), "줄 기록")

## 메모할 것

- 합친 점수가 단일 모델보다 나은지 (전체 / 조용한 / 덜컹거리는):
- mean vs max:
- τ에서 알려진 재질 통과율이 0.95 근처인지, 미등록 거부율이 늘었는지: